<img src=images/xd-logo.png align=right width=300px>

# Async FastAPI


Create a file called `sleepy_api.py` with the code from the cell below (running the cell saves it in the folder of this notebook, `notebooks/fastapi`) and start a server serving the API: in a terminal in that folder, run `uv run uvicorn sleepy_api:app --reload`. And visit http://127.0.0.1:8000/sleep .

When the webserver receives a request it will sleep for the specified ammount of seconds before returning a response.

In [1]:
%%writefile ./sleepy_api.py
from fastapi import FastAPI
import time

app = FastAPI()
 
@app.get("/sleep/{seconds}")
def sleep_for(seconds: int):
    time.sleep(seconds)
    return "Awake!"

Writing ./sleepy_api.py


Now try to request the API multiple times (e.g. by opening the URL multiple times in different tabs) while *the server is sleeping*. What do you expect to happen?

<details>
    
  <summary><span style="color:blue">What is going on?</span></summary>
  
FastAPI will always work asynchronously, since it's build on top of Starlette.

However, if you define APIs that depend on resources that induce some latency (e.g. they are computationally demanding) and they allow for coroutines with `async`and `await`, you can mark your endpoints with `async` and FastaPI will take care of making it work.

</details>

In the below code snippet, the app will sleep for 1 seconds in a for loop. The total sleep seconds depends on how many secondes you put in `sleep_for(seconds: int)`.

In [2]:
%%writefile ./sleepy_api.py
from fastapi import FastAPI
import asyncio
import time

app = FastAPI()

@app.get("/sleepio/{seconds}")
async def sleep_for(seconds: int):
    start = time.time()
    
    print("Going to bed")
    for s in range(seconds+1):
      await asyncio.sleep(1)
      print(f"zz {s}")

    end = time.time()
    print(f"Total sleep time: {end-start} seconds")
    
    return "Awake!"

Overwriting ./sleepy_api.py


### What if we remove `async` -> It is the same!

In [3]:
%%writefile ./sleepy_api.py
from fastapi import FastAPI
import asyncio
import time

app = FastAPI()

@app.get("/sleepio/{seconds}")
def sleep_for(seconds: int):
    start = time.time()
    
    print("Going to bed")
    for s in range(seconds+1):
      time.sleep(1)
      print(f"zz {s}")

    end = time.time()
    print(f"Total sleep time: {end-start} seconds")
    
    return "Awake!"

Overwriting ./sleepy_api.py


### <mark>So why we use async??</mark>

In [4]:
%%writefile ./sleepy_api.py
from fastapi import FastAPI
import asyncio
import time

app = FastAPI()

@app.get("/sleepio/{seconds}")
async def sleep_for(seconds: int):
    start = time.time()

    print("Going to bed")

    tasks = [
        asyncio.create_task(asyncio.sleep(1))
        for s in range(seconds)
    ]

    for s, task in enumerate(tasks):
        await task
        print(f"zz {s}")

    end = time.time()
    print(f"Total sleep time: {end-start} seconds")

    return "Awake!"

Overwriting ./sleepy_api.py


The rules of thumb about when to use `async` endpoints are:   
  - If your endpoint calls functions that support `async`/`await` use `async`.
  - If you endpoint calls functions that don't support `async`/`await`, don't use async.
  - If your endpoint doesn't depend on external resources, use `async` to allow for more concurrent connections to your API.
  - If you are not sure, don't use `async`.